In [ ]:
# ============================================================
# CELL 1: Import all required libraries
# ============================================================

import pandas as pd        # For working with tabular data (like Excel in Python)
import numpy as np         # For numerical operations and math
import matplotlib.pyplot as plt  # For creating charts and graphs
import seaborn as sns      # For beautiful statistical visualizations
import json                # For reading the JSON category file
import warnings            # To suppress unnecessary warning messages

warnings.filterwarnings('ignore')  # Keeps output clean

print("All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

All libraries imported successfully!
Pandas version: 2.2.2
NumPy version: 2.0.2


In [ ]:
# ============================================================
# CELL 2: Load the raw dataset
# ============================================================

# Load the main CSV file
# encoding='latin-1' handles special characters in video titles
df = pd.read_csv('/content/USvideos.csv', encoding='latin-1')

# Load the category JSON file
with open('/content/US_category_id.json', 'r') as f:
    category_data = json.load(f)

# Preview the first 5 rows
print("Dataset loaded successfully!")
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nFirst 5 rows:")
df.head()

Dataset loaded successfully!

Shape: 40949 rows × 16 columns

First 5 rows:


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...
1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13T07:30:00.000Z,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John..."
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO â¶ \n\nSUBSCRIBE âº ...
3,puqaWrEC7tY,17.14.11,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13T11:00:04.000Z,"rhett and link|""gmm""|""good mythical morning""|""...",343168,10172,666,2146,https://i.ytimg.com/vi/puqaWrEC7tY/default.jpg,False,False,False,Today we find out if Link is a Nickelback amat...
4,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095731,132235,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...


In [ ]:
# ============================================================
# CELL 3: Understand the dataset structure
# ============================================================

print("=" * 55)
print("COLUMN NAMES AND DATA TYPES")
print("=" * 55)
print(df.dtypes)

print("\n" + "=" * 55)
print("MISSING VALUES PER COLUMN")
print("=" * 55)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

print("\n" + "=" * 55)
print("BASIC STATISTICS")
print("=" * 55)
df[['views', 'likes', 'dislikes', 'comment_count']].describe()

COLUMN NAMES AND DATA TYPES
video_id                  object
trending_date             object
title                     object
channel_title             object
category_id                int64
publish_time              object
tags                      object
views                      int64
likes                      int64
dislikes                   int64
comment_count              int64
thumbnail_link            object
comments_disabled           bool
ratings_disabled            bool
video_error_or_removed      bool
description               object
dtype: object

MISSING VALUES PER COLUMN
             Missing Count  Missing %
description            570       1.39

BASIC STATISTICS


,views,likes,dislikes,comment_count
count,4.094900e+04,4.094900e+04,4.094900e+04,4.094900e+04
mean,2.360785e+06,7.426670e+04,3.711401e+03,8.446804e+03
std,7.394114e+06,2.288853e+05,2.902971e+04,3.743049e+04
min,5.490000e+02,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.423290e+05,5.424000e+03,2.020000e+02,6.140000e+02
50%,6.818610e+05,1.809100e+04,6.310000e+02,1.856000e+03
75%,1.823157e+06,5.541700e+04,1.938000e+03,5.755000e+03
max,2.252119e+08,5.613827e+06,1.674420e+06,1.361580e+06


In [ ]:
# ============================================================
# CELL 4: Convert category numbers to readable names
# ============================================================

# Extract category id → name mapping from the JSON file
category_map = {}
for item in category_data['items']:
    category_map[int(item['id'])] = item['snippet']['title']

print("Category mapping loaded:")
for k, v in category_map.items():
    print(f"  {k:>3} → {v}")

# Apply the mapping to create a new readable column
df['category_name'] = df['category_id'].map(category_map)

print(f"\nNew column 'category_name' created.")
print(df['category_name'].value_counts().head(10))

Category mapping loaded:
    1 → Film & Animation
    2 → Autos & Vehicles
   10 → Music
   15 → Pets & Animals
   17 → Sports
   18 → Short Movies
   19 → Travel & Events
   20 → Gaming
   21 → Videoblogging
   22 → People & Blogs
   23 → Comedy
   24 → Entertainment
   25 → News & Politics
   26 → Howto & Style
   27 → Education
   28 → Science & Technology
   29 → Nonprofits & Activism
   30 → Movies
   31 → Anime/Animation
   32 → Action/Adventure
   33 → Classics
   34 → Comedy
   35 → Documentary
   36 → Drama
   37 → Family
   38 → Foreign
   39 → Horror
   40 → Sci-Fi/Fantasy
   41 → Thriller
   42 → Shorts
   43 → Shows
   44 → Trailers

New column 'category_name' created.
category_name
Entertainment           9964
Music                   6472
Howto & Style           4146
Comedy                  3457
People & Blogs          3210
News & Politics         2487
Science & Technology    2401
Film & Animation        2345
Sports                  2174
Education               1656
Name:

In [ ]:
# ============================================================
# CELL 5: Fix date and time formatting (timezone-safe version)
# ============================================================

# Convert trending_date from 'YY.DD.MM' format to proper datetime
df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m')

# Convert publish_time — then REMOVE timezone info with .dt.tz_localize(None)
# This is the fix: we strip timezone so both columns are tz-naive
df['publish_time'] = pd.to_datetime(df['publish_time'], utc=True).dt.tz_localize(None)

# Extract useful time features from publish_time
df['publish_hour']        = df['publish_time'].dt.hour
df['publish_day_of_week'] = df['publish_time'].dt.day_name()
df['publish_month']       = df['publish_time'].dt.month
df['publish_year']        = df['publish_time'].dt.year

# Now both columns are tz-naive — subtraction works cleanly
df['days_to_trend'] = (df['trending_date'] - df['publish_time'].dt.normalize()).dt.days

# Clip negative values (a few videos have data entry errors where
# publish_time appears AFTER trending_date — we set those to 0)
df['days_to_trend'] = df['days_to_trend'].clip(lower=0)

print("Date columns fixed successfully!")
print(f"\nSample days_to_trend values:")
print(df['days_to_trend'].describe().round(1))
print(f"\nVideos that trended same day as publishing: "
      f"{(df['days_to_trend'] == 0).sum()}")
print(f"Average days to trend: {df['days_to_trend'].mean():.1f} days")

Date columns fixed successfully!

Sample days_to_trend values:
count    40949.0
mean        16.8
std        146.0
min          0.0
25%          3.0
50%          5.0
75%          9.0
max       4215.0
Name: days_to_trend, dtype: float64

Videos that trended same day as publishing: 121
Average days to trend: 16.8 days


In [ ]:
# ============================================================
# CELL 6: Clean and validate engagement numbers
# ============================================================

# Check for any zero or negative values (data errors)
print("Rows with zero views:", (df['views'] == 0).sum())
print("Rows with zero likes:", (df['likes'] == 0).sum())
print("Rows with negative values:", (df['views'] < 0).sum())

# Remove rows where views = 0 (these are data errors, not real videos)
df = df[df['views'] > 0]

# Create engagement ratio columns — these are KEY analysis metrics
df['like_rate']    = (df['likes']         / df['views'] * 100).round(4)
df['dislike_rate'] = (df['dislikes']      / df['views'] * 100).round(4)
df['comment_rate'] = (df['comment_count'] / df['views'] * 100).round(4)
df['engagement_rate'] = (
    (df['likes'] + df['dislikes'] + df['comment_count'])
    / df['views'] * 100
).round(4)

print("\nNew engagement ratio columns created:")
print(df[['views', 'likes', 'like_rate',
          'comment_rate', 'engagement_rate']].describe().round(4))

Rows with zero views: 0
Rows with zero likes: 172
Rows with negative values: 0

New engagement ratio columns created:
              views         likes   like_rate  comment_rate  engagement_rate
count  4.094900e+04  4.094900e+04  40949.0000    40949.0000       40949.0000
mean   2.360785e+06  7.426670e+04      3.4413        0.4453           4.0480
std    7.394114e+06  2.288853e+05      2.7009        0.5736           3.0243
min    5.490000e+02  0.000000e+00      0.0000        0.0000           0.0000
25%    2.423290e+05  5.424000e+03      1.4967        0.1607           1.9157
50%    6.818610e+05  1.809100e+04      2.8273        0.2961           3.3811
75%    1.823157e+06  5.541700e+04      4.6751        0.5214           5.3799
max    2.252119e+08  5.613827e+06     29.0466       11.7643          33.0255


In [ ]:
# ============================================================
# CELL 7: Clean title and text columns
# ============================================================

# Clean the title column
df['title_clean'] = (
    df['title']
    .str.strip()                          # Remove leading/trailing spaces
    .str.replace(r'[^\x00-\x7F]', '', regex=True)  # Remove non-ASCII (emojis etc.)
    .str.replace(r'\s+', ' ', regex=True) # Replace multiple spaces with one
)

# Create a title length column (character count)
df['title_length'] = df['title_clean'].str.len()

# Clean tags — replace [none] with actual NaN (missing value)
df['tags'] = df['tags'].replace('[none]', np.nan)

# Flag videos with disabled features
df['has_comments'] = ~df['comments_disabled'].astype(bool)
df['has_ratings']  = ~df['ratings_disabled'].astype(bool)

print("Text columns cleaned!")
print(f"\nAverage title length: {df['title_length'].mean():.1f} characters")
print(f"Shortest title: {df['title_length'].min()} characters")
print(f"Longest title:  {df['title_length'].max()} characters")
print(f"\nVideos with tags missing: {df['tags'].isnull().sum()}")

Text columns cleaned!

Average title length: 48.3 characters
Shortest title: 0 characters
Longest title:  100 characters

Videos with tags missing: 1535


In [ ]:
# ============================================================
# CELL 8: Handle duplicate entries
# ============================================================

print(f"Total rows before deduplication: {len(df)}")

# Count how many unique videos appear on multiple trending days
video_trend_counts = df.groupby('video_id')['trending_date'].count()
print(f"Videos that trended multiple days: "
      f"{(video_trend_counts > 1).sum()}")
print(f"Max days a single video trended: {video_trend_counts.max()}")

# Create two versions — both are useful for different analyses
# Version 1: Full dataset (all trending appearances)
df_full = df.copy()

# Version 2: Peak performance only (keep the row with highest views per video)
df_peak = (
    df.sort_values('views', ascending=False)
      .drop_duplicates(subset='video_id', keep='first')
      .reset_index(drop=True)
)

print(f"\nFull dataset rows:  {len(df_full)}")
print(f"Peak dataset rows:  {len(df_peak)}")
print("\nWe will use df_peak for most analyses.")

Total rows before deduplication: 40949
Videos that trended multiple days: 5644
Max days a single video trended: 30

Full dataset rows:  40949
Peak dataset rows:  6351

We will use df_peak for most analyses.


In [ ]:
# ============================================================
# CELL 9: Final summary and save cleaned data
# ============================================================

print("=" * 55)
print("CLEANED DATASET SUMMARY")
print("=" * 55)
print(f"Total unique videos:    {len(df_peak):,}")
print(f"Total columns:          {df_peak.shape[1]}")
print(f"Date range:             "
      f"{df_peak['trending_date'].min().date()} → "
      f"{df_peak['trending_date'].max().date()}")
print(f"Total views in dataset: {df_peak['views'].sum():,}")
print(f"Average views/video:    {df_peak['views'].mean():,.0f}")
print(f"Avg engagement rate:    {df_peak['engagement_rate'].mean():.2f}%")
print(f"Categories present:     {df_peak['category_name'].nunique()}")
print(f"Unique channels:        {df_peak['channel_title'].nunique():,}")

print("\nAll columns in cleaned dataset:")
for col in df_peak.columns:
    print(f"  • {col}")

# Save the cleaned file
df_peak.to_csv('USvideos_cleaned.csv', index=False)
df_full.to_csv('USvideos_full.csv', index=False)
print("\nCleaned files saved!")

CLEANED DATASET SUMMARY
Total unique videos:    6,351
Total columns:          30
Date range:             2017-11-14 → 2018-06-14
Total views in dataset: 12,472,425,372
Average views/video:    1,963,852
Avg engagement rate:    3.53%
Categories present:     16
Unique channels:        2,199

All columns in cleaned dataset:
  • video_id
  • trending_date
  • title
  • channel_title
  • category_id
  • publish_time
  • tags
  • views
  • likes
  • dislikes
  • comment_count
  • thumbnail_link
  • comments_disabled
  • ratings_disabled
  • video_error_or_removed
  • description
  • category_name
  • publish_hour
  • publish_day_of_week
  • publish_month
  • publish_year
  • days_to_trend
  • like_rate
  • dislike_rate
  • comment_rate
  • engagement_rate
  • title_clean
  • title_length
  • has_comments
  • has_ratings

Cleaned files saved!


In [ ]:
# ============================================================
# CELL 10: Data quality report (for README and portfolio)
# ============================================================

print("=" * 55)
print("DATA QUALITY REPORT")
print("=" * 55)

total = len(df_peak)

# Completeness check
print("\nCOMPLETENESS (non-null %)")
print("-" * 35)
key_cols = ['views', 'likes', 'comment_count', 'category_name',
            'publish_time', 'days_to_trend', 'title_clean',
            'engagement_rate']
for col in key_cols:
    pct = (df_peak[col].notnull().sum() / total * 100)
    bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
    print(f"  {col:<20} {bar} {pct:.1f}%")

# Outlier check
print("\nOUTLIER SUMMARY (top 1% vs median)")
print("-" * 35)
for col in ['views', 'likes', 'engagement_rate']:
    median = df_peak[col].median()
    p99    = df_peak[col].quantile(0.99)
    print(f"  {col:<20} median={median:>12,.0f}  "
          f"top 1%={p99:>12,.0f}")

# Category breakdown
print("\nCATEGORY BREAKDOWN")
print("-" * 35)
cat_counts = df_peak['category_name'].value_counts()
for cat, count in cat_counts.items():
    pct = count / total * 100
    print(f"  {cat:<28} {count:>5} videos  ({pct:.1f}%)")

print("\n" + "=" * 55)
print("Dataset is clean and ready for EDA.")
print("=" * 55)

DATA QUALITY REPORT

COMPLETENESS (non-null %)
-----------------------------------
  views                ████████████████████ 100.0%
  likes                ████████████████████ 100.0%
  comment_count        ████████████████████ 100.0%
  category_name        ████████████████████ 100.0%
  publish_time         ████████████████████ 100.0%
  days_to_trend        ████████████████████ 100.0%
  title_clean          ████████████████████ 100.0%
  engagement_rate      ████████████████████ 100.0%

OUTLIER SUMMARY (top 1% vs median)
-----------------------------------
  views                median=     518,107  top 1%=  25,189,198
  likes                median=      11,906  top 1%=     806,986
  engagement_rate      median=           3  top 1%=          14

CATEGORY BREAKDOWN
-----------------------------------
  Entertainment                 1621 videos  (25.5%)
  Music                          801 videos  (12.6%)
  Howto & Style                  594 videos  (9.4%)
  Comedy                       

In [14]:
# ============================================================
# CELL 11: Save cleaned files permanently to Google Drive
# ============================================================

from google.colab import drive

# This mounts your Google Drive into Colab
# A popup will ask you to sign in and give permission — click Allow
drive.mount('/content/drive')

# Create a project folder in your Drive
import os
project_path = '/content/drive/MyDrive/podcast_analytics'
os.makedirs(project_path, exist_ok=True)

# Save both cleaned files to Drive
df_peak.to_csv(f'{project_path}/USvideos_cleaned.csv', index=False)
df_full.to_csv(f'{project_path}/USvideos_full.csv', index=False)

print("Files saved to Google Drive permanently!")
print(f"Location: {project_path}")
print("\nFiles saved:")
print(f"  USvideos_cleaned.csv — {len(df_peak):,} rows")
print(f"  USvideos_full.csv    — {len(df_full):,} rows")

Mounted at /content/drive
Files saved to Google Drive permanently!
Location: /content/drive/MyDrive/podcast_analytics

Files saved:
  USvideos_cleaned.csv — 6,351 rows
  USvideos_full.csv    — 40,949 rows
